In [101]:
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

import anndata as ad

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

from myllia_metric import myllia_score
LN2 = float(np.log(2.0))

In [ ]:
def load_means(means_path):
    df = pd.read_csv(means_path)
    gene_cols = [c for c in df.columns if c != "pert_symbol"]

    base_row = df.loc[df["pert_symbol"].astype(str) == "non-targeting"]
    base = base_row[gene_cols].iloc[0].to_numpy(np.float32)

    tr = df.loc[df["pert_symbol"].astype(str) != "non-targeting"].reset_index(drop=True)
    train_perts = tr["pert_symbol"].astype(str).tolist()
    delta_means = tr[gene_cols].to_numpy(np.float32) - base[None, :]

    return gene_cols, base, train_perts, delta_means

def load_truth_table_wide(gt_path, gene_cols):
    gt_df = pd.read_csv(gt_path)
    gt_df = gt_df[["pert_id"] + gene_cols].copy()
    gt_df["pert_id"] = gt_df["pert_id"].astype(str)

    return gt_df

def load_val_map(valmap_path):
    df = pd.read_csv(valmap_path)

    val_perts = df["pert"].astype(str).tolist()
    val_ids = df["pert_id"].tolist()
    return val_perts, val_ids

In [ ]:
def build_gene_embeddings_from_h5ad(h5ad_path, genes_to_embed, emb_dim, seed):
    adata = ad.read_h5ad(str(h5ad_path))

    genesU = [str(g).upper() for g in genes_to_embed]
    varU = pd.Index([str(v).upper() for v in adata.var_names])

    pos = varU.get_indexer(genesU)
    ok = pos >= 0

    missing = [genesU[i] for i in range(len(genesU)) if not ok[i]]
    if missing:
        print(f"[warn] {len(missing)} / {len(genesU)} genes not found in h5ad.var_names. Example: {missing[:12]}")

    pos_ok = pos[ok]
    genes_ok = [genesU[i] for i in range(len(genesU)) if ok[i]]

    X_counts = adata.X
    X_counts = X_counts.tocsr()

    # Compute per-cell library sizes on all genes
    libsize = np.asarray(X_counts.sum(axis=1)).ravel()
    libsize = np.clip(libsize, 1e-12, None)
    scale = (1e4 / libsize).astype(np.float32)

    # Subset to the genes we want, then apply CP10K normalization
    X_sub = X_counts[:, pos_ok].multiply(scale[:, None])

    # log2(1 + x) on sparse
    X_sub.data = np.log1p(X_sub.data) / LN2

    # Standardize (sparse-safe)
    scaler = StandardScaler(with_mean=False)
    X_sub = scaler.fit_transform(X_sub)

    # SVD fit on cells x genes
    svd = TruncatedSVD(n_components=emb_dim, random_state=seed)
    svd.fit(X_sub)

    # components_: (emb_dim, n_genes_sub) so transpose gives (n_genes_sub, emb_dim)
    Z = svd.components_.T.astype(np.float32)

    gene2z = {genes_ok[i]: Z[i] for i in range(len(genes_ok))}
    return gene2z

In [ ]:
class LowRankHyperNet(nn.Module):
    """
    Predict delta expression:
      delta_pred = (MLP(z_g)) @ B^T
    where:
      z_g: gene embedding (emb_dim,)
      MLP(z_g): coefficients c_g in R^{k_out}
      B: learned basis in R^{G x k_out}
    """
    def __init__(self, n_genes, emb_dim, k_out, hidden, dropout):
        super().__init__()
        self.B = nn.Parameter(torch.randn(n_genes, k_out) * 0.02)

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, k_out),
        )

    def forward(self, z):
        c = self.mlp(z)  # (batch, k_out)
        y = c @ self.B.T # (batch, G)
        return y

In [108]:
def make_df(pert_ids, Y, gene_cols):
    df = pd.DataFrame(Y, columns=gene_cols)
    df.insert(0, "pert_id", [str(x) for x in pert_ids])
    return df

def score_df(solution_df, submission_df, gene_cols):
    """
    Align solution to submission by pert_id, then call myllia_score.
    """
    sol = solution_df[["pert_id"] + gene_cols].copy()
    sub = submission_df[["pert_id"] + gene_cols].copy()

    sub["pert_id"] = sub["pert_id"].astype(str)
    sol["pert_id"] = sol["pert_id"].astype(str)

    sol = sol.set_index("pert_id").loc[sub["pert_id"]].reset_index()

    dt = sol[gene_cols].to_numpy(np.float32)
    dp = sub[gene_cols].to_numpy(np.float32)

    return myllia_score(dt, dp)

In [128]:
def gate_smoothstep(x, a, b):
    """
    Smoothstep gate in torch, matches myllia_metric structure:
      x <= a -> 0
      x >= b -> 1
      else smoothstep((x-a)/(b-a))
    """
    if b <= a:
        raise ValueError("gate_smoothstep requires b > a")
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


def weighted_l1_like_metric(delta_true, delta_pred, eps=1e-12):
    """
    Approximate the metric's gene weighting:
      w_gene = gate(|delta_true|)
      loss = mean_i [ sum_j w_ij * |err_ij| / sum_j w_ij ]
    """
    w = gate_smoothstep(torch.abs(delta_true), a=0.0, b=0.2)
    err = torch.abs(delta_pred - delta_true)
    num = torch.sum(w * err, dim=1)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)
    return torch.mean(num / den)

In [131]:
ROOT = "."
EMB_DIM = 128
K_OUT = 64
HIDDEN = 256
DROPOUT = 0.10
LR = 3e-3
WD = 1e-4
EPOCHS = 1500
FOLDS = 8
SEED = 6
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [113]:
root = Path(ROOT)

means_path = root / "data" / "training_data_means.csv"
gt_path = root / "data" / "training_data_ground_truth_table.csv"
h5ad_path = root / "data" / "training_cells.h5ad"
valmap_path = root / "data" / "pert_ids_val.csv"
out_sub = root / "submissions" / "sub_internal_hypernet.csv"
out_sub.parent.mkdir(parents=True, exist_ok=True)

In [114]:
np.random.seed(SEED)
torch.manual_seed(SEED)

gene_cols, base, train_perts, delta_means = load_means(means_path)
G = len(gene_cols)
N = len(train_perts)
print(f"N_train_perts={N}  G_outputs={G}")

gt_df = load_truth_table_wide(gt_path, gene_cols)

N_train_perts=80  G_outputs=5127


In [116]:
sol_full = gt_df[gt_df["pert_id"].isin([str(p) for p in train_perts])].copy().reset_index(drop=True)
sol_full = sol_full.set_index("pert_id").loc[[str(p) for p in train_perts]].reset_index()
delta_true = sol_full[gene_cols].to_numpy(np.float32)

In [118]:
val_perts, val_ids = load_val_map(valmap_path)

embed_genes = sorted(
    set(g.upper() for g in gene_cols) |
    set(p.upper() for p in train_perts) |
    set(p.upper() for p in val_perts)
)

In [ ]:
gene2z = build_gene_embeddings_from_h5ad(h5ad_path, embed_genes, emb_dim=EMB_DIM, seed=SEED)

In [120]:
Z_train = np.stack([gene2z[p.upper()] for p in train_perts], axis=0).astype(np.float32)

In [125]:
# zero baseline
pids = [str(p) for p in train_perts]
sol_df_full = make_df(pids, delta_true, gene_cols)
sub0 = make_df(pids, np.zeros_like(delta_true), gene_cols)
r0 = score_df(sol_df_full, sub0, gene_cols)
print(f"zero baseline score={r0.score:.6f}  wcos={r0.wcos:.6f}  mean_term={r0.mean_term:.6f}  pred_wmae={r0.pred_wmae:.6f}")

zero baseline score=0.000000  wcos=0.000000  mean_term=0.000000  pred_wmae=0.093218


In [126]:
# mean baseline
mu = delta_true.mean(axis=0, keepdims=True)
subm = make_df(pids, np.repeat(mu, repeats=N, axis=0), gene_cols)
rm = score_df(sol_df_full, subm, gene_cols)
print(f"mean score={rm.score:.6f}  wcos={rm.wcos:.6f}  mean_term={rm.mean_term:.6f}  pred_wmae={rm.pred_wmae:.6f}")

mean score=0.086824  wcos=0.471441  mean_term=0.184168  pred_wmae=0.080940


In [ ]:
device = torch.device(DEVICE)
Zt = torch.tensor(Z_train, device=device)
Yt = torch.tensor(delta_true, device=device)

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_scores = []

print("Starting CV")
for fold, (tr_idx, va_idx) in enumerate(kf.split(Z_train), 1):
    model = LowRankHyperNet(
        n_genes=G, emb_dim=EMB_DIM, k_out=K_OUT,
        hidden=HIDDEN, dropout=DROPOUT
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    best_score = -1e18
    best_state = None

    tr_idx_t = torch.tensor(tr_idx, device=device, dtype=torch.long)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        pred = model(Zt.index_select(0, tr_idx_t))
        loss = weighted_l1_like_metric(Yt.index_select(0, tr_idx_t), pred)

        opt.zero_grad()
        loss.backward()
        opt.step()

        if epoch % 50 == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t)).detach().cpu().numpy().astype(np.float32)

            va_true = delta_true[va_idx]
            va_pids = [pids[i] for i in va_idx]

            sol_va = make_df(va_pids, va_true, gene_cols)
            sub_va = make_df(va_pids, va_pred, gene_cols)

            res = score_df(sol_va, sub_va, gene_cols)
            sc = res.score

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    fold_scores.append(best_score)
    print(f"fold {fold}: best_score={best_score:.6f}")

print(f"cv mean={float(np.mean(fold_scores)):.6f} std={float(np.std(fold_scores)):.6f}")

Starting CV
fold 1: best_score=0.126727
fold 2: best_score=0.097448
fold 3: best_score=0.073580
fold 4: best_score=0.104292
fold 5: best_score=0.128932
fold 6: best_score=0.157297
fold 7: best_score=0.123903
fold 8: best_score=0.052192
cv mean=0.108046 std=0.031417


In [136]:
model = LowRankHyperNet(
    n_genes=G, emb_dim=EMB_DIM, k_out=K_OUT,
    hidden=HIDDEN, dropout=DROPOUT
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

pids = [str(p) for p in train_perts]           # same order as Yt / delta_true
sol_df = make_df(pids, delta_true, gene_cols)  # ground-truth df for scoring

for epoch in range(1500):
    model.train()
    pred = model(Zt)
    loss = weighted_l1_like_metric(Yt, pred)

    opt.zero_grad()
    loss.backward()
    opt.step()

    if (epoch + 1) % 50 == 0:
        model.eval()
        with torch.no_grad():
            pred_np = model(Zt).detach().cpu().numpy().astype(np.float32)

        sub_df = make_df(pids, pred_np, gene_cols)
        res = score_df(sol_df, sub_df, gene_cols)

        print(
            f"Epoch={epoch+1:4d} "
            f"loss={float(loss.item()):.6f} "
            f"score={res.score:.6f} "
            f"wcos={res.wcos:.6f} "
            f"mean_term={res.mean_term:.6f} "
            f"pred_wmae={res.pred_wmae:.6f}"
        )


Epoch=  50 loss=0.068117 score=0.243328 wcos=0.512608 mean_term=0.474685 pred_wmae=0.067961
Epoch= 100 loss=0.054395 score=0.563318 wcos=0.704471 mean_term=0.799633 pred_wmae=0.053730
Epoch= 150 loss=0.045219 score=0.857422 wcos=0.795772 mean_term=1.077472 pred_wmae=0.044478
Epoch= 200 loss=0.039251 score=1.091950 wcos=0.844736 mean_term=1.292652 pred_wmae=0.038142
Epoch= 250 loss=0.033307 score=1.367777 wcos=0.889945 mean_term=1.536922 pred_wmae=0.031882
Epoch= 300 loss=0.029588 score=1.576533 wcos=0.913979 mean_term=1.724911 pred_wmae=0.027642
Epoch= 350 loss=0.027048 score=1.737076 wcos=0.928223 mean_term=1.871398 pred_wmae=0.024822
Epoch= 400 loss=0.025351 score=1.848941 wcos=0.938275 mean_term=1.970576 pred_wmae=0.023089
Epoch= 450 loss=0.024166 score=1.959146 wcos=0.947778 mean_term=2.067094 pred_wmae=0.021565
Epoch= 500 loss=0.023281 score=2.063194 wcos=0.953285 mean_term=2.164299 pred_wmae=0.020174
Epoch= 550 loss=0.022284 score=2.130841 wcos=0.958279 mean_term=2.223613 pred_wm

In [137]:
mu_vec = mu.astype(np.float32).reshape(-1)  # (G,)

Z_val = []
fallback_mask = []
for p in val_perts:
    pu = p.upper()
    if pu in gene2z:
        Z_val.append(gene2z[pu])
        fallback_mask.append(False)
    else:
        Z_val.append(np.zeros((EMB_DIM,), np.float32))
        fallback_mask.append(True)

Z_val = torch.tensor(np.stack(Z_val, axis=0), device=device)

model.eval()
with torch.no_grad():
    Y_val = model(Z_val).detach().cpu().numpy().astype(np.float32)

if any(fallback_mask):
    idxs = [i for i, f in enumerate(fallback_mask) if f]
    print(f"[warn] {len(idxs)} val perts missing embeddings, using mean-delta fallback. Example: {[val_perts[i] for i in idxs[:12]]}")
    for i in idxs:
        Y_val[i] = mu_vec

sub = pd.DataFrame(Y_val, columns=gene_cols)
sub.insert(0, "pert_id", val_ids)
sub.to_csv(out_sub, index=False)
print(f"[ok] wrote: {out_sub}")

[ok] wrote: submissions\sub_internal_hypernet.csv
